# 07 · Evaluasi pada Data Uji
**Bab III Subbab 3.9 dan 3.10.** Notebook ini dijalankan **satu kali** setelah semua keputusan (hiperparameter, ambang, aturan pasangan)
dikunci. Jangan mengubah apa pun di notebook 05–06 berdasarkan hasil di sini.

Isi tabel Bab IV:
1. Cakupan evaluasi (berapa produk dan ulasan yang benar-benar dievaluasi).
2. Perbandingan baseline dan model utama dengan IK 95% (RM1), selisih berpasangan dengan koreksi Holm.
3. Ablasi dan skala data (RM2).
4. Strategi target pada dua protokol relevansi (RM3), per kelompok usia.
5. Analisis kesalahan.

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Untuk menyimpan data dan hasil di Google Drive, lihat petunjuk di `notebooks/settings.py`.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
from hr import experiment as X, report as R, metrics
b = X.load_bundle(cfg)
MS, TG, NB = cfg.main_scale, cfg.main_target, cfg.eval_bootstrap
groups = R.group_runs(X.results_table(cfg))
T = lambda name: cfg.path("tables", name)
list(groups)

## 1. Cakupan evaluasi
Produk yang semua ulasannya bernilai 0 suara tidak bisa dievaluasi dengan NDCG (IDCG = 0). Jumlahnya dilaporkan agar pembaca tahu bagian data yang benar-benar tercakup.

In [ ]:
te = b.rows("test")
pp_u = R.eval_config(b, groups[("U", MS, TG)])
cov = metrics.coverage(te, pp_u)
cov["share_products_evaluated"] = cov["products_ndcg_valid"] / cov["products_total"]
T("uji_cakupan.json").write_text(json.dumps(cov, indent=2)); pd.Series(cov)

## 2. Baseline dan model utama (RM1)

In [ ]:
keys = [(c, MS, TG) for c in ["B0", "B1", "B2", "B3", "B4", "B5", "U"]]
main_tab, pps = R.comparison_table(b, groups, keys, TG, NB)
main_tab.to_csv(T("uji_baseline.csv"), index=False)
main_tab[["code", "seeds", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi", "ndcg@5", "spearman", "map", "n_ndcg@10"]].round(4)

Perbandingan utama yang diuji ketat adalah **U vs B5** (sumbangan pelatihan domain) dan **U vs B1** (BLaIR vs TF-IDF). Sisanya bersifat eksploratif. p-value dikoreksi Holm.

In [ ]:
pair_tab = R.paired_table(pps, ("U", MS, TG), [k for k in keys if k[0] != "U"], "ndcg@10", NB)
pair_tab.to_csv(T("uji_selisih_berpasangan.csv"), index=False); pair_tab.round(4)

In [ ]:
if ("B4", MS, TG) in groups:
    pw = R.pointwise_errors(b, groups[("B4", MS, TG)])
    T("uji_b4_mae_rmse.json").write_text(json.dumps(pw)); print("B4 pada ln(1+v):", pw)

## 3. Ablasi dan skala data (RM2)

In [ ]:
abl_tab, _ = R.comparison_table(b, groups, [(c, MS, TG) for c in ["A1", "A2", "A3", "U"]], TG, NB)
abl_tab["code"] = abl_tab["code"].replace({"U": "A4 (= U)"})
abl_tab.to_csv(T("uji_ablasi.csv"), index=False)
scale_tab, _ = R.comparison_table(b, groups, [("U", s, TG) for s in cfg.scales], TG, NB)
scale_tab["reviews"] = scale_tab["scale"].map(cfg.scales)
scale_tab.to_csv(T("uji_skala.csv"), index=False)
display(abl_tab[["code", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi"]].round(4))
scale_tab[["scale", "reviews", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi"]].round(4)

## 4. Strategi target (RM3)
Setiap model dievaluasi pada **dua protokol relevansi**: mentah (T1) dan terkoreksi usia (T3). Angka dari dua protokol dilaporkan di tabel terpisah dan tidak dibandingkan langsung.

In [ ]:
tkeys = [("U", MS, t) for t in ("T1", "T2", "T3")]
tt = []
for proto in ("T1", "T3"):
    tab, _ = R.comparison_table(b, groups, tkeys, proto, NB)
    tab["protokol_relevansi"] = proto; tt.append(tab)
target_tab = pd.concat(tt, ignore_index=True)
target_tab.to_csv(T("uji_target.csv"), index=False)
target_tab[["protokol_relevansi", "target", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi", "spearman", "map"]].round(4)

In [ ]:
edges = json.loads(cfg.path("age_group_edges.json").read_text())
ag = R.age_groups(b, edges)
subsets = {g: te[ag.loc[te.index] == g] for g in ["baru", "menengah", "lama"]}
rows = []
for k in tkeys:
    if k in groups:
        for proto in ("T1", "T3"):
            t = R.by_subset(b, groups[k], subsets, proto); t["target"] = k[2]; t["protokol_relevansi"] = proto
            rows.append(t)
age_tab = pd.concat(rows, ignore_index=True); age_tab.to_csv(T("uji_per_kelompok_usia.csv"), index=False)
age_tab.round(4)

## 5. Kinerja menurut ukuran produk dan panjang ulasan (model U)

In [ ]:
size = te.groupby("parent_asin").size()
size_grp = pd.qcut(size.rank(method="first"), 4, labels=["Q1 kecil", "Q2", "Q3", "Q4 besar"])
d = pp_u.merge(size_grp.rename("kelompok").reset_index(), on="parent_asin")
size_tab = d.groupby("kelompok", observed=True)[["ndcg@10", "spearman", "ap"]].mean().round(4)
size_tab.to_csv(T("uji_per_ukuran_produk.csv"))
lens = pd.qcut(te["word_count"], 3, labels=["pendek", "sedang", "panjang"])
len_tab = R.by_subset(b, groups[("U", MS, TG)], {g: te[lens == g] for g in ["pendek", "sedang", "panjang"]}, TG)
len_tab.to_csv(T("uji_per_panjang_ulasan.csv"), index=False)
display(size_tab); len_tab.round(4)

## 6. Analisis kesalahan
Produk dengan NDCG@10 terendah untuk model U, beserta ulasan yang ditaruh paling atas oleh model dan yang seharusnya paling atas.

In [ ]:
worst = pp_u.dropna(subset=["ndcg@10"]).nsmallest(10, "ndcg@10")
worst.to_csv(T("uji_produk_terburuk.csv"), index=False)
pred = X.load_pred(cfg, groups[("U", MS, TG)][0], "test")
ex = te.merge(pred, on="rid")
for p in worst["parent_asin"].head(3):
    g = ex[ex.parent_asin == p]
    print("=" * 100, "\nproduk", p, "| ulasan:", len(g))
    print("TOP MODEL :", g.nlargest(1, "score")[["helpful_vote", "word_count", "title"]].to_dict("records"))
    print("TOP SUARA :", g.nlargest(1, "helpful_vote")[["helpful_vote", "word_count", "title"]].to_dict("records"))
worst